# Practical — Transient thermal fin: POD-greedy, then observations

**One thermal-fin notebook, two teaching stages.** Start with **A — Prediction and POD-greedy** after [the stationary greedy practical](https://feelpp.github.io/course-rom/course-rom/labs/session04-greedy.html). Stop at its checkpoint for today&#8217;s work. **B — Sequential observations** reuses the same model later, after KF/EnKF; it is not a prerequisite for part A. The historical session-13 URL is retained so existing links and downloads still work.


Download the notebook from the toolbar and [thermal-fin-coarse.npz](https://feelpp.github.io/course-rom/course-rom/_attachments/data/thermal-fin-coarse.npz); keep them beside each other. Use [the Python environment](https://feelpp.github.io/course-rom/course-rom/setup.html), including SciPy. The model and solver follow the opening of [the original parabolic problem set](https://feelpp.github.io/course-rom/course-rom/_attachments/problem-set-parabolic.pdf); this short practical does not reproduce its long derivations or implementation exercises. Read [the model notes](https://feelpp.github.io/course-rom/rom/applications/thermal-fin.html) and [the parabolic extension](https://feelpp.github.io/course-rom/rom/certification/extensions.html#parabolic-prediction). The full-order solver, prediction API, POD helper and diagnostics are supplied. Focus on the concepts: observe trajectories, compress them, select useful trajectories, and see how errors change with dimension. Only the short adaptive loop is left to implement; detailed certification and filter work are optional reading.


The Python variable `n` represents the full dimension $\mathcal N$; `r` represents the reduced dimension $N$ in the formulas. See [the common notation](https://feelpp.github.io/course-rom/rom/notation.html).
## A — Prediction and POD-greedy (75 minutes)

## 1. Explore the supplied 2D solver (15 minutes)

There are $\mathcal N=1333$ finite-element temperature unknowns. The five material conductivities are fixed at one; the positive Biot number $Bi\in[0.01,1$] controls Robin boundary heat loss. The spatial problem is two-dimensional, but **the parameter domain here is one-dimensional**. With a known zero initial state,

$$
M\dot u+A(Bi)u=f\,g(t),\quad
A(Bi)=A_0+Bi A_5,\quad A_0=\sum_{q=0}^4 A_q.
$$
As in the original exercise, use $T=10$, $\Delta t=0.1$ and train with the discrete impulse $g(t_k)=\delta_{1k}$. The forcing vector is the root-boundary flux load, not a volumetric heat source. The output follows the PDF: $s(t)=\int_\Omega u(t)\simeq\ell^Tu(t)$, with $\ell=M\mathbf1$. Divide by the area $|\Omega|=\mathbf1^TM\mathbf1$ to obtain the spatial mean; this normalization does not change the reduction method. Test another control, $g(t)=1-\cos t$, at $Bi=0.05$.
The PDF requests the medium mesh. Here its original **coarse** matrices (1 333 unknowns) keep the demonstration quick; the same method applies to the medium mesh. The full [data archive](https://feelpp.github.io/course-rom/course-rom/_attachments/homework-2024-data.zip) remains available for later exploration. No mesh-refinement study is required today.
Backward Euler uses the **same** time grid for full and reduced models:

$$
(M+\Delta t A)u^k=Mu^{k-1}+\Delta t f g(t_k),\qquad
(M_N+\Delta t A_N)a^k=M_ra^{k-1}+\Delta t f_N g(t_k).
$$
First run the full-order solver and inspect its finite-element solutions. For a fixed parameter and time step, factor the time-stepping matrix once per trajectory. Reduced operators are introduced only after this first visualization.


In [ ]:
import numpy as np
from pathlib import Path
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import splu
from scipy.linalg import lu_factor, lu_solve
from time import perf_counter

path = Path('thermal-fin-coarse.npz')
if not path.exists(): path = Path('docs/modules/ROOT/attachments/data/thermal-fin-coarse.npz')
if not path.exists(): raise FileNotFoundError('Download thermal-fin-coarse.npz beside this notebook.')
data = np.load(path, allow_pickle=False)
def matrix(name):
    return csc_matrix((data[name+'_data'], data[name+'_indices'], data[name+'_indptr']),
                      shape=tuple(data[name+'_shape']))
M = matrix('M'); Aq = [matrix(f'A{i}') for i in range(6)]
A0 = sum(Aq[:5]); f = data['f']; coordinates = data['coordinates']; n = len(f)
def operator(Bi): return A0+Bi*Aq[5]
dt = .1; steps = 100; times = dt*np.arange(steps+1)
inputs = np.zeros(steps); inputs[0] = 1.  # Original discrete impulse.
prediction_inputs = 1-np.cos(times[1:])
ell = np.asarray(M@np.ones(n))  # PDF output: integral, not area-normalized mean.
area = ell.sum()

def trajectory(Bi, control=None):
    control = inputs if control is None else np.asarray(control)
    solve = splu(M+dt*operator(Bi)).solve
    states = [np.zeros(n)]
    for g in control:
        states.append(solve(M@states[-1]+dt*f*g))
    return np.column_stack(states)

print(f'Transient thermal fin: {n} DOFs; impulse training; {steps} time steps.')


## Compute the finite-element examples

Use the smooth root flux $g(t)=1-\cos t$ to see the fin heat up. Plot three Biot numbers to observe the influence of boundary heat loss. The spatial panels below show the same $Bi=0.05$ case that will later be compared with the reduced model. POD training will use the impulse input defined above.
This cell performs the numerical work and stores fields and outputs. To explore the impulse response instead, set `fe_control = inputs` and rerun this computation cell before plotting.


In [ ]:
fe_control = prediction_inputs.copy()
fe_parameters = (.01, .05, 1.)
fe_display_Bi = .05
fe_display_steps = np.array([10, 50, 100])  # t=1, 5, 10.
fe_states = {Bi: trajectory(Bi, fe_control) for Bi in fe_parameters}
fe_outputs = {Bi: ell@states for Bi, states in fe_states.items()}
fe_fields = fe_states[fe_display_Bi][:, fe_display_steps].copy()
fe_field_times = times[fe_display_steps]
print('Finite-element examples computed: fields, outputs and input are ready to plot.')


## Visualize the stored finite-element solutions

No solver is called in this cell. The top row shows temperature fields at three instants on a common colour scale; the bottom row shows the output of interest and the imposed root flux. Change plot formatting without rerunning the numerical cell. All results here come from the full finite-element model; POD and reduced prediction have not been used yet.


In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(12, 7), layout='constrained')
grid = fig.add_gridspec(2, 3)
fe_axes = [fig.add_subplot(grid[0, j]) for j in range(3)]
fe_output_axis = fig.add_subplot(grid[1, :2])
fe_input_axis = fig.add_subplot(grid[1, 2])
for ax, field, time in zip(fe_axes, fe_fields.T, fe_field_times):
    fe_image = ax.tripcolor(coordinates[:, 0], coordinates[:, 1], data['triangles'], field,
                            shading='gouraud', vmin=0., vmax=fe_fields.max())
    ax.set(xlabel='x', ylabel='y', aspect='equal', title=f'FE temperature at t={time:g}')
fig.colorbar(fe_image, ax=fe_axes, label='Temperature; common scale', shrink=.8)
for Bi, values in fe_outputs.items():
    fe_output_axis.plot(times, values, label=f'Bi={Bi:g}')
for time in fe_field_times:
    fe_output_axis.axvline(time, color='gray', linestyle=':', linewidth=.7)
fe_output_axis.set(xlabel='Time', ylabel='Integrated temperature',
                   title='Finite-element output: s(t)=ell.T @ u(t)')
fe_output_axis.legend()
fe_input_axis.plot(times[1:], fe_control)
fe_input_axis.set(xlabel='Time', ylabel='Root heat-flux amplitude g(t)', title='Applied input')
for ax in (fe_output_axis, fe_input_axis):
    ax.grid(True, alpha=.25)
fig.suptitle(f'Full finite-element model: {n} unknowns; spatial fields at Bi={fe_display_Bi:g}')
plt.show()


**Task 1.** Compare the three finite-element output curves and explain the influence of Biot number. Relate the input, output and spatial fields. Change Bi, the displayed instants or the input, rerun the numerical cell, then replot. The supplied LU factorization is reused at each time step; no implementation of a PDE solver is required.
## Reduced prediction API (numerics only)

Now introduce the small prediction model. Project mass, affine stiffness components, source and output once per basis. The integrator below evolves only reduced coefficients; reconstruction is separate. This cell defines numerical functions and parameter sets, and produces no plot.


In [ ]:
def project_fin(Z):
    return dict(M=Z.T@(M@Z), A0=Z.T@(A0@Z), A5=Z.T@(Aq[5]@Z),
                f=Z.T@f, ell=ell@Z)

def predict_fin(reduced, Bi, control=None):
    control = inputs if control is None else np.asarray(control)
    Mr = reduced['M']; Ar = reduced['A0']+Bi*reduced['A5']
    factor = lu_factor(Mr+dt*Ar)
    propagation = lu_solve(factor, Mr)
    source = lu_solve(factor, dt*reduced['f'])
    a = np.zeros(len(reduced['f'])); coefficients = [a.copy()]
    for g in control:
        a = propagation@a+source*g
        coefficients.append(a.copy())
    return np.column_stack(coefficients)

def mass_rmse(error):
    return float(np.sqrt(np.mean(np.sum(error*(M@error), axis=0))))

def max_mass_error(error):
    return float(np.sqrt(np.maximum(0., np.sum(error*(M@error), axis=0))).max())

train_Bi = np.geomspace(.01, 1., 9)
test_Bi = np.sqrt(train_Bi[:-1]*train_Bi[1:])


## 2. Supplied helpers: compress a trajectory and assess its error (10 minutes)

Use the **mass norm** for trajectory POD: it measures spatial temperature error. Equal weights $w_k=\Delta t/T=1/K$ on the noninitial time steps measure the time-mean squared projection error. The known zero initial snapshot need not enter POD.
For certification use a **different fixed metric**, $G=A(Bi_{\rm ref})$, $Bi_{\rm ref}=0.1$. The nonnegative affine components give $\alpha_{\rm LB}(Bi)=\min(1,Bi/Bi_{\rm ref})$. This is a lower bound relative to $G$, not relative to $M$.
The residual includes the time derivative:

$$
\mathbf r_\mu^k=f g(t_k)-MZ\frac{a^k-a^{k-1}}{\Delta t}-A(Bi)Za^k,\qquad
D(Bi)=\left(\frac{\Delta t}{\alpha_{\rm LB}(Bi)}
\sum_{k=1}^K(\mathbf r_\mu^k)^TG^{-1}\mathbf r_\mu^k\right)^{1/2}.
$$
**Reading only; implementation is supplied.** With identical zero initial states, the [discrete energy estimate](https://feelpp.github.io/course-rom/rom/certification/extensions.html#transient-bound) gives $\max_k\|u^{\mathcal N}^k-Za^k\|_M\le D(Bi)$. Indeed each partial residual sum is bounded by the full sum. This is a conservative **trajectory** certificate, not the stationary residual bound applied independently to each time step. For nonzero initial mismatch, add its squared mass norm under the square root.


In [ ]:
Bi_ref = .1
G = operator(Bi_ref)
riesz_solve = splu(G).solve

def scan_fin(Z, parameters):
    reduced = project_fin(Z)
    MZ = M@Z; A0Z = A0@Z; A5Z = Aq[5]@Z
    bounds = []
    for Bi in parameters:
        a = predict_fin(reduced, Bi)
        rho = (f[:, None]*inputs[None, :]
               - MZ@np.diff(a, axis=1)/dt
               - (A0Z+Bi*A5Z)@a[:, 1:])
        representatives = riesz_solve(rho)
        residual_squared = np.maximum(0., np.sum(rho*representatives, axis=0))
        alpha_lb = min(1., Bi/Bi_ref)
        bounds.append(np.sqrt(dt*np.sum(residual_squared)/alpha_lb))
    return np.asarray(bounds)

# scan_fin calls no full-order trajectory. Direct residual/Riesz evaluation
# still depends on n; an offline residual factor is advanced work.
def leading_mass_mode(E):
    # Weighted method of snapshots: w_k=1/steps on this uniform grid.
    correlation = E.T@(M@E)/E.shape[1]
    values, vectors = np.linalg.eigh(correlation)
    if values[-1] <= 1e-24:
        raise ValueError('No resolved trajectory defect remains.')
    return E@vectors[:, -1]/np.sqrt(E.shape[1]*values[-1])

training_trajectories = {}  # Clear if changing the training input or time grid.

def mass_pod_modes(defects, ric_target=.999, max_modes=None):
    """Complete the time-weighted mass POD and return (modes, diagnostics)."""
    # R1: form the correlation matrix in the mass metric (uniform time weights).
    # R2: compute and sort eigenpairs; identify numerically resolved directions.
    # R3: accumulate eigenvalues and select the smallest rank reaching the RIC.
    # R4: respect max_modes and report whether the target was actually reached.
    # R5: reconstruct and mass-normalize the spatial modes.
    # Treat a zero defect separately: its RIC ratio is undefined.
    raise NotImplementedError('Complete the defect POD / RIC steps with the instructor')

def enrich_trajectory(Z, Bi):
    if float(Bi) not in training_trajectories:
        training_trajectories[float(Bi)] = trajectory(Bi)
    states = training_trajectories[float(Bi)][:, 1:]
    defect = states-Z@(Z.T@(M@states))
    v = leading_mass_mode(defect)
    for _ in range(2):
        v -= Z@(Z.T@(M@v))
    norm = np.sqrt(max(0., v@(M@v)))
    if norm <= 1e-10:
        raise ValueError('Dependent enrichment: inspect POD and stopping thresholds.')
    return np.column_stack([Z, v/norm])


## Defect POD and Relative Information Content (RIC)

For one selected parameter, the trajectory matrix $S=[u^1,\ldots,u^K$] has one spatial temperature field per column. With $Z^TMZ=I$, form the missing part $E=S-Z(Z^TMS)$. A POD mode of $E$ is a spatial pattern not yet represented by the current basis; it is not a time instant.
On this uniform time grid, the correlation matrix is $C_E=E^TME/K$. If its eigenvalues are $\lambda_1\ge\cdots\ge\lambda_d>0$, define

$$
\operatorname{RIC}_E(q)=\frac{\sum_{i=1}^q\lambda_i}{\sum_{i=1}^d\lambda_i},\qquad
\frac{J_q}{J_0}=1-\operatorname{RIC}_E(q),\qquad
\frac{\sqrt{J_q}}{\sqrt{J_0}}=\sqrt{1-\operatorname{RIC}_E(q)}.
$$
Complete `mass_pod_modes` in the indicated steps; its body is intentionally left blank. Choose the smallest resolved rank reaching the target, capped by the remaining basis budget. A RIC of 99% allows 10% relative RMS projection error; 99.9% allows approximately 3.16%; a 1% relative RMS target requires 99.99% RIC. Do not divide by zero when the defect vanishes, and check `target_met` if a rank or resolution limit prevents the requested RIC.
This local RIC concerns projection of one defect trajectory in the mass metric. It is **not** the stopping tolerance for the transient prediction certificate, and does not certify new parameters or new controls. The recorded example below deliberately keeps **one mode per enrichment** so the first implementation is easy to trace. The RIC variant is a second experiment, not a different interpretation of those one-mode plots.
**Task 2.** Identify a trajectory column, its projection defect and a retained spatial mode. Work in short stages: (R1) form the correlation matrix and check its shape; (R2) sort its eigenvalues; (R3) choose a rank for targets 0.99, 0.999 and 0.9999; (R4) reconstruct the modes and check mass orthonormality. Stop after each stage for discussion. Use the discarded eigenvalue tail to define `projection_rms`. Keep the given Riesz and bound routines: no certification implementation is required today.
## 3. Observe trajectory enrichment and implement POD-greedy (35 minutes)

The outer greedy selects a **Biot number**. The inner POD selects one missing **spatial direction from the entire time trajectory**. A selected Biot number may appear again: its trajectory usually contains more than one independent direction. Reuse that cached full trajectory instead of solving it again; count distinct full trajectories separately from enrichments. Thus "one new point" here means one selected trajectory/enrichment, not necessarily a previously unseen parameter.
The following fixed sequence is a recorded example driven by the supplied certificate, with seed $Bi=0.01$. It makes the diagnostics executable before you implement the selection loop. Reference trajectories for plots are collected separately, after basis construction; they are not inputs to the certificate scan.
Compute the trajectory enrichments and diagnostic arrays first.


In [ ]:
recorded_Bi = [0.01, 0.01, 0.01, 0.01, 0.01, 0.01]
Z = np.empty((n, 0)); fin_bases = []; fin_bounds = []
for Bi in recorded_Bi:
    Z = enrich_trajectory(Z, Bi)
    fin_bases.append(Z.copy())
    fin_bounds.append(scan_fin(Z, train_Bi))
# Diagnostic references only; not used by scan_fin or enrichment selection.
fin_truth = {float(Bi): trajectory(Bi) for Bi in np.r_[train_Bi, test_Bi]}
fin_rows = []
for Z, bounds in zip(fin_bases, fin_bounds):
    reduced = project_fin(Z)
    projection = []; train_error = []; test_error = []; output_error = []
    for Bi in np.r_[train_Bi, test_Bi]:
        truth = fin_truth[float(Bi)]
        forecast = Z@predict_fin(reduced, Bi)
        if Bi in train_Bi:
            defect = truth[:, 1:]-Z@(Z.T@(M@truth[:, 1:]))
            projection.append(mass_rmse(defect))
            train_error.append(max_mass_error(truth-forecast))
        else:
            test_error.append(max_mass_error(truth-forecast))
            output_error.append(np.max(np.abs(ell@(truth-forecast))))
    fin_rows.append(dict(rank=Z.shape[1], projection=max(projection),
                         train=max(train_error), test=max(test_error),
                         bound=max(bounds), output=max(output_error)))
print('Thermal-fin POD-greedy diagnostics:')
for row in fin_rows:
    print(f"  r={row['rank']}: max training M-error={row['train']:.3e}; "
          f"trajectory bound={row['bound']:.3e}; max test M-error={row['test']:.3e}")


Plot the stored diagnostics below; this cell does not compute trajectories, POD modes or certificates.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.7))
ranks = [row['rank'] for row in fin_rows]
for key, label in [('projection', 'Max training projection time-RMS'),
                   ('train', 'Training ROM: max in time'), ('test', 'Validation ROM: max in time')]:
    axes[0].semilogy(ranks, [row[key] for row in fin_rows], 'o-', label=label)
axes[1].semilogy(ranks, [row['train'] for row in fin_rows], 'o-', label='Max training M-error')
axes[1].semilogy(ranks, [row['bound'] for row in fin_rows], 's--', label='Max trajectory certificate')
axes[2].plot(ranks, recorded_Bi, 'o-')
for rank, Bi in zip(ranks, recorded_Bi):
    axes[2].annotate(str(rank), (rank, Bi), xytext=(4, 5), textcoords='offset points')
for ax in axes:
    ax.set_xlabel('Basis dimension / trajectory enrichment'); ax.grid(True, alpha=.25)
axes[0].set_ylabel('Mass-norm error'); axes[0].legend(fontsize=6)
axes[1].set_ylabel('Mass-norm quantity'); axes[1].legend(fontsize=7)
axes[2].set(ylabel='Selected Bi', yscale='log', title='Repeated parameters are allowed')
fig.tight_layout(); plt.show()


The **projection** time-RMS cannot increase as the mass-orthonormal space grows. The integrated ROM follows a dynamical equation; unlike the stationary SPD energy projection, its error is not guaranteed to decrease at every enrichment. Look at its measured evolution and the certificate separately, using the same parameters and time grid at each rank.


In [ ]:
def pod_greedy(parameters, seed=.01, tolerance=.05, rank_max=8):
    Z = enrich_trajectory(np.empty((n, 0)), seed)
    selected = [seed]
    history = []
    # TODO: scan_fin at every rank; record max certificate and selected Bi.
    # TODO: test tolerance, then rank budget; report final status.
    # TODO: select the maximum certificate, without reference trajectories.
    # TODO: enrich_trajectory at that parameter and repeat.
    # Second experiment: use mass_pod_modes on the trajectory defect,
    # capped by rank_max-Z.shape[1], then reorthonormalize each new mode.
    raise NotImplementedError('Implement the outer POD-greedy loop')


**Task 3 (core activity).** Keep your first notebook version and complete its greedy function progressively. First reproduce one iteration manually: inspect the scan, choose its maximum, then enrich. Pause to inspect the selected parameter, then implement the loop. Add stopping and budget tests only after the first two enrichments work. Explain why a POD of the **defect**, not the unchanged trajectory, avoids repeatedly choosing directions already in the basis. Then replace one-mode enrichment by the RIC rule above and compare the number of retained modes with the number of selected trajectories. Reorthonormalize the new modes against Z, respect the remaining rank budget, and report an unmet RIC target instead of silently claiming success. Changing the seed is optional. Separate number of selected trajectories from final spatial dimension.
## 4. See the full and reduced solutions together (15 minutes)

This 2D transient example complements the stationary 3D fin studied in MLOps. Here the data are full-order snapshots and the predictive model comes from projecting the governing equations; the time evolution and boundary forcing remain explicit. Compare the spatial fields as well as the output: a good scalar prediction alone does not establish that the whole field is accurate.
Change `display_step` below and rerun the cell to inspect another instant. The full and reduced temperature plots share the same color scale. The error field has its own signed scale. The two bottom panels compare the original PDF output and its error over the entire trajectory, under the new control.
Compute the full and reduced trajectories and store their fields, outputs and errors.


In [ ]:
Zall = fin_bases[-1]
r = Zall.shape[1]; Z = Zall
truth_Bi = .05; forecast_Bi = .12
truth = trajectory(truth_Bi, prediction_inputs)
full_forecast = trajectory(forecast_Bi, prediction_inputs)
reduced = project_fin(Z)
coeff_prediction = predict_fin(reduced, truth_Bi, prediction_inputs)
reduced_field = Z@coeff_prediction  # Reconstruction for visualization, not online output.
comparison_error_field = truth-reduced_field
full_output = ell@truth
rom_output = reduced['ell']@coeff_prediction  # No full field needed for this output.
output_error_history = np.abs(full_output-rom_output)
output_error = output_error_history.max()
output_scale = np.max(np.abs(full_output))
print('Held-out output max error:', output_error)
print(f'Full model: {n} unknowns per step; reduced model: {r}.')
print('Distinct full training trajectories:', len(training_trajectories))
print(f'New-control output error / peak full output: {100*output_error/output_scale:.2f}%')


Plot the stored comparison below. Changing `display_step` only selects stored columns and does not rerun either solver.


In [ ]:
display_step = 50  # Change only this instant and rerun this plotting cell.
full_field = truth[:, display_step]
rom_field = reduced_field[:, display_step]
field_error = comparison_error_field[:, display_step]

fig = plt.figure(figsize=(13, 7), layout='constrained')
grid = fig.add_gridspec(2, 3)
field_axes = [fig.add_subplot(grid[0, j]) for j in range(3)]
output_axis = fig.add_subplot(grid[1, :2])
error_axis = fig.add_subplot(grid[1, 2])
vmin = min(full_field.min(), rom_field.min())
vmax = max(full_field.max(), rom_field.max())
for ax, field, title in zip(field_axes[:2], (full_field, rom_field),
                             (f'Finite elements: {n} DOFs', f'Reduced model: {r} coefficients')):
    temperature_image = ax.tripcolor(coordinates[:, 0], coordinates[:, 1], data['triangles'],
                                     field, shading='gouraud', vmin=vmin, vmax=vmax)
    ax.set(xlabel='x', ylabel='y', aspect='equal')
    ax.set_title(title, fontsize=10)
error_limit = max(np.max(np.abs(field_error)), 1e-14)
error_image = field_axes[2].tripcolor(coordinates[:, 0], coordinates[:, 1], data['triangles'],
                                     field_error, shading='gouraud', cmap='RdBu_r',
                                     vmin=-error_limit, vmax=error_limit)
field_axes[2].set(xlabel='x', ylabel='y', aspect='equal')
field_axes[2].set_title('Finite elements minus reduced field', fontsize=10)
fig.colorbar(temperature_image, ax=field_axes[:2], label='Temperature; shared scale', shrink=.8)
fig.colorbar(error_image, ax=field_axes[2], label='Signed field error', shrink=.8)
output_axis.plot(times, full_output, label='Finite-element output')
output_axis.plot(times, rom_output, '--', label='Reduced output')
output_axis.axvline(times[display_step], color='gray', linestyle=':', label='Displayed field time')
output_axis.set(xlabel='Time', ylabel='Integrated temperature', title='New control: g(t)=1-cos(t); Bi=0.05')
output_axis.legend(fontsize=8)
error_axis.plot(times, output_error_history)
error_axis.set(xlabel='Time', ylabel='Absolute output error')
for ax in (output_axis, error_axis):
    ax.grid(True, alpha=.25)
fig.suptitle(f'Full and reduced thermal fin at t={times[display_step]:g}')
plt.show()


Compare the number of unknowns per step and the relative output discrepancy under the new control. What has the basis learned from the impulse responses?
Submit the rank-error plot, selected parameters, final stopping status, and a full/reduced field comparison and output plot at a held-out Biot number. Record distinct construction-only full trajectories (including cache reuse) separately from diagnostic reference trajectories. Explain which part of the algorithm uses parameters, which uses time snapshots, and which quantities can be precomputed. **Stop here for the prediction/POD-greedy session.**
The same affine operators make prediction efficient. A non-affine coefficient or nonlinear term would break this simple online assembly: this motivates [EIM](https://feelpp.github.io/course-rom/rom/hyper-reduction/index.html) and [DEIM](https://feelpp.github.io/course-rom/rom/hyper-reduction/deim.html). These methods approximate the troublesome term; its POD basis need not be the state basis used here.
## B — Sequential observations (later, after EnKF; 15 minutes plus homework)

Continue in this notebook when studying sequential assimilation. No new thermal model or duplicate prediction practical is needed. Run part A first. The filter&#8217;s forecast uses a deliberately imperfect parameter $Bi=0.12$, while synthetic observations come from $Bi=0.05$. This separates model mismatch from reduction error. The process-noise scale is a declared modeling choice.


In [ ]:
targets = np.array([[0., .1], [-2., 1.], [2., 2.], [0., 3.8]])
sensor_indices = np.array([np.argmin(np.linalg.norm(coordinates-p, axis=1)) for p in targets])
noise_sd = .005
observations = truth[sensor_indices, 1:]+np.random.default_rng(130).normal(0, noise_sd, (4, steps))


In [ ]:
def reduced_filter(Z,observations,sensor_ids,ensemble_size=40,seed=131,noise_sd=.005):
    # No truth or full-state reconstruction is used in this online routine.
    rank=Z.shape[1]; Mr=Z.T@(M@Z); Ar=Z.T@(operator(forecast_Bi)@Z)
    F=np.linalg.solve(Mr+dt*Ar,Mr); b=np.linalg.solve(Mr+dt*Ar,dt*(Z.T@f))
    Hr=Z[sensor_ids,:]; R=noise_sd**2*np.eye(len(sensor_ids))
    rng=np.random.default_rng(seed)
    # Known initial state is zero; process noise models imperfect forecasts.
    X=np.zeros((rank,ensemble_size)); a=np.zeros(rank)
    means=[X.mean(axis=1)]; open_loop=[a.copy()]; spreads=[np.zeros(rank)]
    start=perf_counter()
    for g,y in zip(prediction_inputs,observations.T):
        a=F@a+b*g
        X=F@X+b[:,None]*g+.005*rng.standard_normal(X.shape)
        Y=Hr@X
        DX=X-X.mean(axis=1,keepdims=True); DY=Y-Y.mean(axis=1,keepdims=True)
        Cxy=DX@DY.T/(ensemble_size-1); Cyy=DY@DY.T/(ensemble_size-1)
        gain=np.linalg.solve(Cyy+R,Cxy.T).T
        X=X+gain@(y[:,None]+rng.normal(0,noise_sd,Y.shape)-Y)
        means.append(X.mean(axis=1)); spreads.append(X.std(axis=1,ddof=1)); open_loop.append(a.copy())
    return np.array(means).T,np.array(open_loop).T,np.array(spreads).T,perf_counter()-start

if Zall.shape[1]<r: raise ValueError('Requested rank exceeds resolved snapshot space')
Z=Zall[:,:r]
coeff,open_coeff,spread,online_seconds=reduced_filter(Z,observations,sensor_indices)
# Full fields are reconstructed only for diagnostic plots, outside the online timer.
estimate=Z@coeff; reduced_forecast=Z@open_coeff
print('Thermal-fin mass RMSE:')
print('Full forecast only:',mass_rmse(full_forecast[:,1:]-truth[:,1:]))
print('Reduced forecast only:',mass_rmse(reduced_forecast[:,1:]-truth[:,1:]))
print('Reduced EnKF:',mass_rmse(estimate[:,1:]-truth[:,1:]))
print('Reduction of forecast model:',mass_rmse(reduced_forecast[:,1:]-full_forecast[:,1:]))
print('Online EnKF seconds (all updates; excludes setup/reconstruction):',online_seconds)
assimilation_outputs = dict(truth=ell@truth, full_forecast=ell@full_forecast,
                            reduced_forecast=ell@reduced_forecast, estimate=ell@estimate)


Compare the imperfect full forecast, reduced forecast and reduced EnKF. Which comparison isolates reduction error? Change the assumed noise level only after recording the baseline. Regenerate observations consistently if changing the actual noise.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,3.7))
axes[0].plot(times,assimilation_outputs['truth'],label='Truth')
axes[0].plot(times,assimilation_outputs['full_forecast'],'--',label='Full forecast, imperfect Bi')
axes[0].plot(times,assimilation_outputs['reduced_forecast'],':',label='Reduced forecast')
axes[0].plot(times,assimilation_outputs['estimate'],label='Reduced EnKF')
axes[0].set(xlabel='Time',ylabel='Integrated temperature'); axes[0].legend(fontsize=7)
im=axes[1].tripcolor(coordinates[:,0],coordinates[:,1],data['triangles'],estimate[:,-1],shading='gouraud')
axes[1].scatter(coordinates[sensor_indices,0],coordinates[sensor_indices,1],c='black',marker='x',label='Sensors')
axes[1].set(xlabel='x',ylabel='y',title='Final assimilated temperature',aspect='equal')
fig.colorbar(im,ax=axes[1]); fig.tight_layout(); plt.show()


**Later-session checkpoint.** Submit the output plot and separate errors for parameter mismatch, model reduction and assimilation. Explain why coefficient-space noise and ensemble spread cannot represent every full-state discrepancy. Continue with [the sequential-assimilation homework](https://feelpp.github.io/course-rom/course-rom/homework/04-sequential-assimilation.html).
